In [ ]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [ ]:
import comet_ml
import torch
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
from copy import deepcopy
import numpy as np
import gymnasium as gym
from itertools import product
from tqdm import tqdm
from RL4CRN.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN.Environments.CRNEnvironment import CRNEnvironment
from RL4CRN.Environments.VecCRNEnvironment import VecCRNEnvironment
from RL4CRN.Agents.RecurrentAgent import RecurrentAgent
from RL4CRN.Input_Output_Rxn_Networks.CRNGenerator import CRNCompletor
from RL4CRN.Rewards.Transients import dynamic_tracking_error

In [ ]:
# Set the logger to use Comet
api_key = "o77J6VCMDamustkfJuMXZ2jdV"
logger = CometLogger(
    api_key=api_key,
    project="Testing_Rendering",        
    workspace="maurice-filo" 
)
logger = logger.experiment

In [ ]:
# Construct a basic CRN
species_labels = ['X_1', 'X_2', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2', 'u_3']
stoichiometry_reactants = np.array([[0], [0], [1], [0]], dtype=np.int8)
stoichiometry_products = np.array([[1], [0], [1], [0]], dtype=np.int8)
parameters = np.array([1], dtype=np.float32)
input_influence_matrix = np.array([[0], [0], [0]], dtype=np.int8)
outputs = np.array([2], dtype=np.int8)
IOCRN_AIF = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)

In [ ]:
# Add reactions
k_1 = 1; gamma_1 = 1; gamma_2 = 0.1; mu = 10; theta = np.nan; eta = 0.1
IOCRN_AIF.add_reaction({'reactants index': 1, 'products index': 6, 'input influence index': 0, 'rate constant': k_1})
IOCRN_AIF.add_reaction({'reactants index': 1, 'products index': 0, 'input influence index': 1, 'rate constant': gamma_1})
IOCRN_AIF.add_reaction({'reactants index': 2, 'products index': 0, 'input influence index': 2, 'rate constant': gamma_2})
IOCRN_AIF.add_reaction({'reactants index': 0, 'products index': 3, 'input influence index': 3, 'rate constant': mu})
IOCRN_AIF.add_reaction({'reactants index': 2, 'products index': 11, 'input influence index': 0, 'rate constant': theta})
IOCRN_AIF.add_reaction({'reactants index': 13, 'products index': 0, 'input influence index': 0, 'rate constant': eta})
print('Initial CRN:')
IOCRN_AIF.print_reactions()

In [ ]:
# Create an environment
max_num_reactions = 2
N_CPUs = 128
n_samples = 10*N_CPUs
CRN_template = deepcopy(IOCRN_AIF)
vec_env = VecCRNEnvironment([CRNEnvironment(CRN_template, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], N_CPUs=N_CPUs, logger=logger)

In [ ]:
# Specify the parameters
num_species = 4
num_reactions = 2
num_inputs = 3
allow_input_influence = False

width = 256
depth = 3

# Define RSG_Attributes
class RSG_Attributes:
    def __init__(self, width, depth, allow_input_influence):
        self.LSTM_hidden_size = width
        self.FFNN_hidden_size = [width, width, width] if allow_input_influence else [width, width]
        self.FFNN_num_layers = [depth, depth, depth] if allow_input_influence else [depth, depth]
        self.weight = [None, None, None] if allow_input_influence else [None, None]

# Define PSG_Attributes
class PSG_Attributes:
    def __init__(self, width, depth):
        self.LSTM_hidden_size = width
        self.FFNN_hidden_size = width
        self.FFNN_num_layers = 3
        self.weight = None

In [ ]:
# Construct the completor
device = 'cuda' if torch.cuda.is_available() else 'cpu'
n_grid = 200
param_1 = np.linspace(0.01, 20, n_grid, dtype=np.float32)
param_2 = np.linspace(0.01, 20, n_grid, dtype=np.float32)
param_3 = np.linspace(0.01, 20, n_grid, dtype=np.float32)
parameter_grid = np.stack([param_1, param_2, param_3], axis=0)

rsg_attributes = RSG_Attributes(width, depth, allow_input_influence)
psg_attributes = PSG_Attributes(width, depth)
completor = CRNCompletor(num_species, num_reactions, IOCRN_AIF.num_unknown_parameters, num_inputs, parameter_grid, n_samples, rsg_attributes, psg_attributes, device=device, allow_input_influence=allow_input_influence).to(device)

In [ ]:
# Create a recurrent agent
agent = RecurrentAgent(vec_env.envs[0], completor, allow_input_influence=False, logger=logger, learning_rate=1e-4, entropy_weight=100, entropy_update_coefficient=0.95, entropy_schedule=10, minimum_entropy_weight=1, risk=0.95, risk_update=0.00, max_risk=1.00, risk_schedule=20)

In [ ]:
# Create the reward function
def compute_reward(state):
    nums = [0.5, 1, 1.5]
    u = np.array(list(product(nums, repeat=len(nums))), dtype=np.float32)
    initial_condition = np.array([0, 0, 0, 0], dtype=np.float32)
    time_horizon = np.linspace(0, 150, 1000, dtype=np.float32)
    r = u[:,2] * state.parameters[4]
    return dynamic_tracking_error(state, u, initial_condition, time_horizon, r, threshold=1000)

In [ ]:
# Run the forward pass
epoch_num = 1000
render_schedule = 5
for i in tqdm(range(epoch_num)):
    vec_env.reset()
    for j in range(max_num_reactions + vec_env.envs[0].CRN_template.num_unknown_parameters):
        actions = agent.act()
        out = vec_env.step(actions)
    rewards = vec_env.get_reward(compute_reward)
    agent.update(rewards)
    if i % render_schedule == 0:
        vec_env.render(rewards, mode='logger_image')

  3%|▎         | 31/1000 [00:50<26:56,  1.67s/it]